### Set Env

In [1]:
import sys
import importlib
from pathlib import Path

# 1. Cari PROJECT_ROOT: walk up dari cwd sampai ketemu requirements.txt
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "requirements.txt tidak ditemukan. Jalankan notebook dari dalam folder project."
    )

# 2. Tambah PROJECT_ROOT ke sys.path (untuk `import src.set_env` dan `import src.rag`)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. WAJIB: invalidate cache supaya Python melihat sys.path yang baru
importlib.invalidate_caches()

# 4. Sanity check — pastikan file-nya ada
set_env_file = PROJECT_ROOT / "src" / "set_env.py"
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"set_env.py   : {set_env_file}  (ada? {set_env_file.is_file()})")
assert set_env_file.is_file(), f"File tidak ada di {set_env_file}"

# 5. Import & jalankan
from src.set_env import check_and_setup_environment

result = check_and_setup_environment(project_root=PROJECT_ROOT, auto_install=True)
print("\nResult:", result)

PROJECT_ROOT : /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
set_env.py   : /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/src/set_env.py  (ada? True)
Python  : 3.12.16
Interp. : /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/.venv/bin/python
Root    : /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
reqfile : /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/requirements.txt
Total   : 11 package (1 notebook-only)

── Cek Awal ──
Package                 Min        Terpasang      Status
---------------------------------------------------------
pandas                  2.2        3.1.0          ✅ ok
numpy                   1.26  

# FASE 1 — Bagian A: RAG (Rencana 7 Langkah)

### 1. Load Chunks

In [2]:
#setup Path

import sys
from pathlib import Path
import re

# Notebook ada di notebook/, project root ada di parent-nya
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

# Supaya `from src.rag import ...` bisa jalan
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Cek lokasi
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Docs ada?", (PROJECT_ROOT / "data" / "rag" / "docs").exists())

PROJECT_ROOT: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
Python: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/.venv/bin/python
Docs ada? True


### Baca Docstring & Test Asli

In [3]:
import re
import importlib
from pathlib import Path

# ─────────────────────────────────────────────
# Baca docstring load_chunks dari src/rag.py
# ─────────────────────────────────────────────
rag_src = (PROJECT_ROOT / "src" / "rag.py").read_text(encoding="utf-8")

m = re.search(r"def load_chunks\(.*?\n(    \"\"\".*?\"\"\")", rag_src, re.DOTALL)
print("=" * 70)
print("DOCSTRING load_chunks (dari src/rag.py)")
print("=" * 70)
print(m.group(1) if m else "(tidak ditemukan — cek manual src/rag.py)")

# ─────────────────────────────────────────────
# Baca test yang menyebut load_chunks
# ─────────────────────────────────────────────
test_src = (PROJECT_ROOT / "tests" / "test_rag.py").read_text(encoding="utf-8")

print("\n" + "=" * 70)
print("TEST yang menyebut load_chunks")
print("=" * 70)

# Split test file per fungsi (def test_...)
blocks = re.split(r"(?=^def test_)", test_src, flags=re.MULTILINE)
for block in blocks:
    if "load_chunks" in block:
        print(block.strip())
        print("-" * 70)

DOCSTRING load_chunks (dari src/rag.py)
    """Potong setiap dokumen .md per baris 'Pasal n: ...' (document-aware chunking)."""

TEST yang menyebut load_chunks
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from rag import answer, load_chunks, rrf  # noqa: E402
----------------------------------------------------------------------
def test_chunking_per_pasal():
    chunks = load_chunks()
    assert len(chunks) == 24                      # 8 dokumen x 3 pasal
    assert chunks[0]["chunk_id"] == "K01-P1"
----------------------------------------------------------------------


#### Melihat Data Asli

In [4]:
#Melihat Data Asli

docs_dir = PROJECT_ROOT / "data" / "rag" / "docs"

files = sorted(docs_dir.glob("K*.md"))
print(f"Jumlah dokumen: {len(files)}\n")

for f in files:
    print(f"--- {f.name} ---")
    print(f.read_text(encoding="utf-8"))
    print()

Jumlah dokumen: 8

--- K01.md ---
# K01 — Syarat Pemohon Kredit
Pasal 1: Pemohon kredit tanpa agunan (KTA) harus berusia 21 sampai 60 tahun pada saat pengajuan.
Pasal 2: Pemohon wajib memiliki KTP yang masih berlaku dan nomor telepon aktif atas nama sendiri.
Pasal 3: Pemohon karyawan harus memiliki penghasilan tetap minimal 6 bulan di tempat kerja saat ini.


--- K02.md ---
# K02 — Plafon dan Tenor
Pasal 1: Plafon KTA paling tinggi Rp200.000.000 per nasabah.
Pasal 2: Tenor kredit dapat dipilih antara 12 sampai 60 bulan.
Pasal 3: Total cicilan seluruh pinjaman nasabah tidak boleh melebihi 30% dari penghasilan bulanan (debt burden ratio).


--- K03.md ---
# K03 — Suku Bunga
Pasal 1: Suku bunga efektif KTA berkisar 11% sampai 16% per tahun, ditentukan oleh skor risiko pemohon.
Pasal 2: Nasabah yang menerima gaji melalui rekening Bank Nusantara (payroll) mendapat potongan bunga 1% per tahun.
Pasal 3: Suku bunga bersifat tetap selama tenor dan tidak berubah walaupun suku bunga acuan naik.



#### Uji Regex di Satu File

In [5]:
sample = (docs_dir / "K06.md").read_text(encoding="utf-8")

# Regex untuk match "Pasal N: teks..."
pattern = r"Pasal\s+(\d+):\s*(.+)"

matches = list(re.finditer(pattern, sample))
print(f"Jumlah match di K06: {len(matches)}  (target: 3)\n")

for m in matches:
    print(f"  pasal={m.group(1)}  |  text={m.group(2)[:70]}...")

Jumlah match di K06: 3  (target: 3)

  pasal=1  |  text=Keterlambatan pembayaran cicilan dikenakan denda 0,1% per hari dari ci...
  pasal=2  |  text=Kredit yang menunggak lebih dari 90 hari digolongkan sebagai kredit ma...
  pasal=3  |  text=Nasabah yang mengalami kesulitan bayar dapat mengajukan restrukturisas...


In [6]:
import importlib
import subprocess
import sys
import src.rag

# ─────────────────────────────────────────────────────────
# Reload modul (WAJIB)
# ─────────────────────────────────────────────────────────
# Jupyter menyimpan cache modul di memori. Tanpa reload, 
# perubahan yang baru kamu save di src/rag.py tidak akan terbaca.
importlib.reload(src.rag)
from src.rag import load_chunks

# ─────────────────────────────────────────────────────────
# Panggil fungsi
# ─────────────────────────────────────────────────────────
docs_dir = PROJECT_ROOT / "data" / "rag" / "docs"
chunks = load_chunks(docs_dir)

print(f"Total chunk dihasilkan: {len(chunks)}")
print(f"Chunk pertama : {chunks[0]['chunk_id']} | {chunks[0]['title']}")
print(f"Chunk terakhir: {chunks[-1]['chunk_id']} | {chunks[-1]['title']}")
print("-" * 50)



Total chunk dihasilkan: 24
Chunk pertama : K01-P1 | K01 — Syarat Pemohon Kredit
Chunk terakhir: K08-P3 | K08 — Perlindungan Data dan Pengaduan
--------------------------------------------------


In [7]:
# ─────────────────────────────────────────────────────────
# Verifikasi Otomatis (Assertion Manual)
# ─────────────────────────────────────────────────────────
try:
    assert len(chunks) == 24, f"Harus 24 chunk, dapat {len(chunks)}"
    assert chunks[0]["chunk_id"] == "K01-P1"
    assert chunks[0]["pasal"] == 1
    assert isinstance(chunks[0]["pasal"], int)
    assert chunks[-1]["chunk_id"] == "K08-P3"
    print("✅ Assertion manual lolos: Struktur dan jumlah data sudah benar.")
except AssertionError as e:
    print(f"❌ Assertion manual GAGAL: {e}")

✅ Assertion manual lolos: Struktur dan jumlah data sudah benar.


In [8]:
# ─────────────────────────────────────────────────────────
# Jalankan Pytest langsung dari Notebook
# ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🔄 Menjalankan pytest -k test_chunking_per_pasal ...")
print("="*60 + "\n")

# Menjalankan pytest sebagai subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-k", "test_chunking_per_pasal", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

# Tampilkan output standar dari pytest
print(result.stdout)

# Pengecekan hasil (exit code 0 berarti sukses di pytest)
if result.returncode == 0:
    print("🎉 PYTEST LULUS! Fungsi load_chunks sudah sesuai standar penilai.")
    print("👉 Langkah selanjutnya: Step 1.2 (tokenize & _index).")
else:
    print("⚠️ PYTEST GAGAL.")
    if result.stderr:
        print("STDERR:", result.stderr)
    print("Cek pesan error di atas untuk melihat detail assertion yang gagal.")


🔄 Menjalankan pytest -k test_chunking_per_pasal ...

============================= test session starts ==============================
platform linux -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
plugins: anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 3 items / 2 deselected / 1 selected

tests/test_rag.py::test_chunking_per_pasal PASSED                        [100%]

======================= 1 passed, 2 deselected in 4.71s ========================

🎉 PYTEST LULUS! Fungsi load_chunks sudah sesuai standar penilai.
👉 Langkah selanjutnya: Step 1.2 (tokenize & _index).


### Tokenize and Index

In [9]:
import importlib
import time
import src.rag

# Reload modul
importlib.reload(src.rag)
from src.rag import tokenize, _index

print("=== 1. Uji fungsi tokenize ===")
hasil_token = tokenize("NPWP, wajib!")
print(f"Input : 'NPWP, wajib!'")
print(f"Output: {hasil_token}")
assert hasil_token == ["npwp", "wajib"], "Tokenize gagal!"
print("✅ Tokenize benar.\n")


print("=== 2. Uji fungsi _index ===")
print("⏳ Memuat model dan membuat index... (Harap tunggu, ini bisa memakan waktu 5-20 menit jika baru pertama kali download).")

t0 = time.time()
chunks, texts, embedder, vectors, bm25, reranker = _index()
waktu = time.time() - t0

print(f"\n✅ Index berhasil dibuat dalam {waktu:.2f} detik.")
print(f"Jumlah chunks : {len(chunks)}")
print(f"Jumlah texts  : {len(texts)}")
print(f"Shape vectors : {vectors.shape} (Harusnya: 24 x 384)")
print(f"Tipe embedder : {type(embedder).__name__}")
print(f"Tipe bm25     : {type(bm25).__name__}")
print(f"Tipe reranker : {type(reranker).__name__}")

# Verifikasi dimensi vektor
assert vectors.shape == (24, 384), "Dimensi vektor salah!"
assert len(texts) == 24
print("\n🎉 Step 1.2 Selesai! Model berhasil dimuat dan index siap digunakan.")

=== 1. Uji fungsi tokenize ===
Input : 'NPWP, wajib!'
Output: ['npwp', 'wajib']
✅ Tokenize benar.

=== 2. Uji fungsi _index ===
⏳ Memuat model dan membuat index... (Harap tunggu, ini bisa memakan waktu 5-20 menit jika baru pertama kali download).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ Index berhasil dibuat dalam 10.30 detik.
Jumlah chunks : 24
Jumlah texts  : 24
Shape vectors : (24, 384) (Harusnya: 24 x 384)
Tipe embedder : SentenceTransformer
Tipe bm25     : BM25Okapi
Tipe reranker : CrossEncoder

🎉 Step 1.2 Selesai! Model berhasil dimuat dan index siap digunakan.


#### dense_search dan bm25_search

##### 1. Dense Search (Si Pencari Makna)
* Cara Kerja: Mengubah teks menjadi vektor (angka) dan menghitung kemiripan matematika (Cosine Similarity).
* Kekuatan: Sangat pintar memahami sinonim, konteks, dan beda bahasa. (Contoh: Mencari "telat" bisa menemukan "keterlambatan").
* Kelemahan: Sering "terlalu pintar" hingga mengabaikan angka atau istilah spesifik. (Contoh: Bingung membedakan "denda 0,1%" dan "penalti 2%").


##### 2. BM25 Search (Si Pencari Kata Persis)
* Cara Kerja: Memecah teks menjadi kata (tokenize) dan mencari kecocokan kata yang persis sama (Exact Match). Memberi bobot lebih pada kata yang jarang muncul.
* Kekuatan: Sangat presisi untuk angka dan istilah khusus. (Contoh: Mencari "NPWP" atau "90 hari" akan langsung ketemu).
* Kelemahan: Sangat kaku. Jika user bertanya "telat", ia tidak akan menemukan dokumen yang berisi kata "keterlambatan".


In [10]:
import importlib
import src.rag

# Reload modul
importlib.reload(src.rag)
from src.rag import dense_search, bm25_search, _index

# Panggil _index() sekali agar masuk ke cache (sekarang instan)
chunks, texts, embedder, vectors, bm25, reranker = _index()

question = "Berapa denda kalau telat bayar cicilan?"
print(f"Pertanyaan: '{question}'\n")

# ─────────────────────────────────────────────────────────
# Uji Dense Search
# ─────────────────────────────────────────────────────────
dense_indices, dense_scores = dense_search(question)
print("=== Dense Search (Top 3) ===")
for i in range(3):
    idx = dense_indices[i]
    score = dense_scores[i]
    chunk = chunks[idx]
    print(f"Rank {i+1}: [{chunk['chunk_id']}] Score: {score:.4f} | {chunk['text'][:60]}...")

print("\n")

# ─────────────────────────────────────────────────────────
# Uji BM25 Search
# ─────────────────────────────────────────────────────────
bm25_indices = bm25_search(question)
print("=== BM25 Search (Top 3) ===")
for i in range(3):
    idx = bm25_indices[i]
    chunk = chunks[idx]
    # Ambil skor dari BM25 langsung untuk diprint
    raw_score = bm25.get_scores(__import__('src.rag', fromlist=['tokenize']).tokenize(question))[idx]
    print(f"Rank {i+1}: [{chunk['chunk_id']}] Score: {raw_score:.4f} | {chunk['text'][:60]}...")

# ─────────────────────────────────────────────────────────
# Verifikasi
# ─────────────────────────────────────────────────────────
top_dense_id = chunks[dense_indices[0]]['chunk_id']
top_bm25_id = chunks[bm25_indices[0]]['chunk_id']

print("\n" + "="*50)
if top_dense_id.startswith("K06") and top_bm25_id.startswith("K06"):
    print("🎉 BERHASIL! Kedua metode mencari berhasil menemukan pasal yang tepat (K06).")
else:
    print(f"⚠️ Perhatikan: Dense menemukan {top_dense_id}, BM25 menemukan {top_bm25_id}.")

print("👉 Langkah selanjutnya: Step 1.4 (Hybrid RRF).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Pertanyaan: 'Berapa denda kalau telat bayar cicilan?'

=== Dense Search (Top 3) ===
Rank 1: [K06-P1] Score: 0.6114 | Keterlambatan pembayaran cicilan dikenakan denda 0,1% per ha...
Rank 2: [K07-P2] Score: 0.4079 | Pelunasan dipercepat dikenakan penalti 2% dari sisa pokok pi...
Rank 3: [K03-P2] Score: 0.3766 | Nasabah yang menerima gaji melalui rekening Bank Nusantara (...


=== BM25 Search (Top 3) ===
Rank 1: [K06-P1] Score: 4.7329 | Keterlambatan pembayaran cicilan dikenakan denda 0,1% per ha...
Rank 2: [K06-P3] Score: 4.2060 | Nasabah yang mengalami kesulitan bayar dapat mengajukan rest...
Rank 3: [K05-P1] Score: 2.1801 | Model skor risiko menghitung probabilitas gagal bayar pemoho...

🎉 BERHASIL! Kedua metode mencari berhasil menemukan pasal yang tepat (K06).
👉 Langkah selanjutnya: Step 1.4 (Hybrid RRF).


#### Reciprocal Rank Fusion (RRF).

In [11]:
import importlib
import subprocess
import sys
import src.rag

# Reload modul
importlib.reload(src.rag)
from src.rag import rrf, dense_search, bm25_search, _index

print("=== 1. Uji RRF dengan data dummy (sesuai PDF) ===")
# Contoh dari modul: dense [A,B,C], bm25 [B,C,X,Y,A]
# Kita gunakan huruf sebagai representasi indeks chunk
dense_dummy = ["A", "B", "C"]
bm25_dummy  = ["B", "C", "X", "Y", "A"]

hasil_rrf = rrf([dense_dummy, bm25_dummy])
print(f"Dense : {dense_dummy}")
print(f"BM25  : {bm25_dummy}")
print(f"Hasil : {hasil_rrf}")
assert hasil_rrf[:3] == ["B", "C", "A"], "Urutan RRF tidak sesuai harapan!"
print("✅ Logika RRF manual benar.\n")

print("="*50)
print("=== 2. Uji RRF dengan data asli (Dense + BM25) ===")
question = "Berapa denda kalau telat bayar cicilan?"

# Ambil peringkat asli
dense_indices, _ = dense_search(question)
bm25_indices = bm25_search(question)

# Gabungkan dengan RRF
hybrid_indices = rrf([dense_indices, bm25_indices])

# Tampilkan Top 3
chunks, *_ = _index()
print("Hybrid Top 3:")
for i in range(3):
    idx = hybrid_indices[i]
    print(f"Rank {i+1}: [{chunks[idx]['chunk_id']}] {chunks[idx]['text'][:60]}...")

# ─────────────────────────────────────────────────────────
# Jalankan Pytest untuk memastikan sesuai standar template
# ─────────────────────────────────────────────────────────
print("\n" + "="*50)
print("🔄 Menjalankan pytest -k test_rrf_contoh_modul ...")
print("="*50 + "\n")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-k", "test_rrf_contoh_modul", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("🎉 PYTEST LULUS! Fungsi rrf sudah 100% benar.")
else:
    print("⚠️ PYTEST GAGAL. Cek output di atas.")

=== 1. Uji RRF dengan data dummy (sesuai PDF) ===
Dense : ['A', 'B', 'C']
BM25  : ['B', 'C', 'X', 'Y', 'A']
Hasil : ['B', 'C', 'A', 'X', 'Y']
✅ Logika RRF manual benar.

=== 2. Uji RRF dengan data asli (Dense + BM25) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Hybrid Top 3:
Rank 1: [K06-P1] Keterlambatan pembayaran cicilan dikenakan denda 0,1% per ha...
Rank 2: [K06-P2] Kredit yang menunggak lebih dari 90 hari digolongkan sebagai...
Rank 3: [K07-P2] Pelunasan dipercepat dikenakan penalti 2% dari sisa pokok pi...

🔄 Menjalankan pytest -k test_rrf_contoh_modul ...

============================= test session starts ==============================
platform linux -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
plugins: anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 3 items / 2 deselected / 1 selected

tests/test_rag.py::test_rrf_contoh_modul PASSED                          [100%]

======================= 1 passed, 2 deselected in 10.94s ===========

#### Uji Answer

In [12]:
import importlib
import src.rag
importlib.reload(src.rag)

from src.rag import answer

print("=== 1. Uji Pertanyaan Kosong (Harus Fallback Instan) ===")
res = answer("")
print(f"Answer    : {res['answer']}")
print(f"Fallback  : {res['fallback']}")
assert res['fallback'] is True
assert res['latency_ms'] < 10, "Harusnya instan, tidak memuat model"
print("✅ Lolos.\n")

print("=== 2. Uji Pertanyaan Di Luar Topik (Harus Fallback) ===")
# Pertanyaan ini tidak ada di dokumen kebijakan kredit
res = answer("Siapa pemenang Piala Dunia 2022?")
print(f"Answer    : {res['answer']}")
print(f"Top Score : {res['top_score']:.4f}")
print(f"Fallback  : {res['fallback']}")
assert res['fallback'] is True
print("✅ Lolos.\n")

print("=== 3. Uji Pertanyaan Valid (Harus Dijawab + Sitasi) ===")
res = answer("Berapa denda kalau telat bayar cicilan?", mode="hybrid_rerank")
print(f"Answer    : {res['answer']}")
print(f"Sources   : {res['sources']}")
print(f"Top Score : {res['top_score']:.4f}")
print(f"Fallback  : {res['fallback']}")
print(f"Latency   : {res['latency_ms']:.2f} ms")
assert res['fallback'] is False
assert "[Sumber: K06, Pasal 1]" in res['answer']
print("✅ Lolos.\n")

print("=== 4. Uji Mode Lain (Dense / BM25 / Hybrid) ===")
for m in ["dense", "bm25", "hybrid"]:
    res = answer("Berapa denda kalau telat bayar cicilan?", mode=m)
    print(f"Mode {m:<12}: {res['answer'][:50]}...")
print("✅ Lolos.\n")

print("🎉 Semua skenario jawaban berjalan dengan baik!")

=== 1. Uji Pertanyaan Kosong (Harus Fallback Instan) ===
Answer    : Maaf, informasi tersebut tidak ada di dokumen kebijakan kredit.
Fallback  : True
✅ Lolos.

=== 2. Uji Pertanyaan Di Luar Topik (Harus Fallback) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Answer    : Maaf, informasi tersebut tidak ada di dokumen kebijakan kredit.
Top Score : 0.1862
Fallback  : True
✅ Lolos.

=== 3. Uji Pertanyaan Valid (Harus Dijawab + Sitasi) ===
Answer    : Keterlambatan pembayaran cicilan dikenakan denda 0,1% per hari dari cicilan yang tertunggak. [Sumber: K06, Pasal 1]
Sources   : ['K06', 'K02', 'K07']
Top Score : 0.6114
Fallback  : False
Latency   : 39.53 ms
✅ Lolos.

=== 4. Uji Mode Lain (Dense / BM25 / Hybrid) ===
Mode dense       : Keterlambatan pembayaran cicilan dikenakan denda 0...
Mode bm25        : Keterlambatan pembayaran cicilan dikenakan denda 0...
Mode hybrid      : Keterlambatan pembayaran cicilan dikenakan denda 0...
✅ Lolos.

🎉 Semua skenario jawaban berjalan dengan baik!


#### pytest

In [13]:
import subprocess
import sys

print("🔄 Menjalankan SEMUA test di tests/test_rag.py ...")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-v"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)
print(result.stdout)

if result.returncode == 0:
    print("🎉 LUAR BIASA! Seluruh 3 test Bagian A LULUS (PASSED).")
else:
    print("⚠️ Ada test yang gagal. Cek output di atas.")

🔄 Menjalankan SEMUA test di tests/test_rag.py ...
============================= test session starts ==============================
platform linux -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david
plugins: anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 3 items

tests/test_rag.py::test_chunking_per_pasal PASSED                        [ 33%]
tests/test_rag.py::test_rrf_contoh_modul PASSED                          [ 66%]
tests/test_rag.py::test_pertanyaan_kosong_fallback PASSED                [100%]

============================== 3 passed in 7.17s ===============================

🎉 LUAR BIASA! Seluruh 3 test Bagian A LULUS (PASSED).


In [14]:
import importlib
import src.rag
import src.rag_eval

# Reload modul (opsional jika belum ada perubahan, tapi aman untuk memastikan)
importlib.reload(src.rag)
importlib.reload(src.rag_eval)

# Jalankan fungsi evaluasi
src.rag_eval.main()

Melakukan pemanasan model (warm-up)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Memproses mode: dense...
Memproses mode: bm25...
Memproses mode: hybrid...
Memproses mode: hybrid_rerank...

✅ Detail evaluasi disimpan ke: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/reports/rag_eval_detail.csv
✅ Summary evaluasi disimpan ke: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/reports/rag_eval_summary.csv

HASIL EVALUASI RAG
         mode  hit_at_3  citation_acc  fallback_acc  avg_latency_ms
        dense    0.9167        0.8333           1.0           10.57
         bm25    0.8333        0.5833           1.0            8.11
       hybrid    0.9167        0.8333           1.0            7.62
hybrid_rerank    1.0000        1.0000           1.0           19.27


# FASE 2 — Bagian B: MLflow

### 1. Eksplorasi Dataset

##### Membuka File

In [15]:
import importlib
import pandas as pd
import src.data
importlib.reload(src.data)

from src.data import load_data, split, data_version, validate_record, FEATURES

print("=== 1. Uji load_data ===")
df = load_data()
print(f"Total baris: {len(df)}")
print(f"Jumlah 'bad': {df['bad'].sum()}")
print(f"Tipe data 'job': {df['job'].dtype}")

assert len(df) == 1000
assert "bad" in df.columns
assert df["saving_accounts"].isna().sum() == 0, "Masih ada NA di saving_accounts!"

# PERBAIKAN: Cek apakah isinya benar-benar string, abaikan tipe 'object' kaku
assert all(isinstance(x, str) for x in df["job"]), "Job harus berupa teks (string)!"
print("✅ load_data OK.\n")

=== 1. Uji load_data ===
Total baris: 1000
Jumlah 'bad': 300
Tipe data 'job': str
✅ load_data OK.



In [16]:
from sklearn.model_selection import train_test_split
from src.data import FEATURES, TARGET, SEED


X = df[FEATURES]
y = df[TARGET]

X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

print(f"Train : {len(X_train_80)} baris")
print(f"Test : {len(X_test_20)} baris")

Train : 700 baris
Test : 300 baris


### 2. Melatih Model (Logistic Regression, Random Forest, Gradient Boosting)

In [17]:
import importlib
import pandas as pd
import src.data
import src.train

# Reload modul
importlib.reload(src.data)
importlib.reload(src.train)

from src.data import load_data, split
from src.train import build_model, evaluate, CANDIDATES

print("=== Memuat & Split Data ===")
df = load_data()
X_train, X_test, y_train, y_test = split(df)
print(f"Data train: {X_train.shape} | Data test: {X_test.shape}\n")

print("=== Melatih 3 Model ===")
hasil = {}
for name in CANDIDATES.keys():
    print(f"Melatih model: {name} ...")
    model = build_model(name)
    
    # Latih model
    model.fit(X_train, y_train)
    
    # Evaluasi
    metrics = evaluate(model, X_test, y_test)
    hasil[name] = metrics
    
    print(f"  -> AUC: {metrics['roc_auc']} | Recall: {metrics['recall_bad']} | F1: {metrics['f1_bad']}")

print("\n=== Tabel Hasil Evaluasi ===")
df_hasil = pd.DataFrame(hasil).T
print(df_hasil.to_string())

print("\n🎉 Tahap 2 (Train) Selesai!")

=== Memuat & Split Data ===
Data train: (700, 9) | Data test: (300, 9)

=== Melatih 3 Model ===
Melatih model: logreg ...
  -> AUC: 0.7592 | Recall: 0.7222 | F1: 0.5909
Melatih model: random_forest ...
  -> AUC: 0.7629 | Recall: 0.7222 | F1: 0.583
Melatih model: gradient_boosting ...
  -> AUC: 0.767 | Recall: 0.3556 | F1: 0.4324

=== Tabel Hasil Evaluasi ===
                   roc_auc  recall_bad  precision_bad  f1_bad
logreg              0.7592      0.7222         0.5000  0.5909
random_forest       0.7629      0.7222         0.4887  0.5830
gradient_boosting   0.7670      0.3556         0.5517  0.4324

🎉 Tahap 2 (Train) Selesai!


### 3.Quality Gate

In [18]:
import importlib
import json
from pathlib import Path
import src.quality_gate

# Reload modul
importlib.reload(src.quality_gate)
from src.quality_gate import check, GATES

print("=== 1. Uji Fungsi `check` ===")
# Skenario 1: Model bagus
lulus_1, alasan_1 = check({"roc_auc": 0.76, "recall_bad": 0.72})
print(f"Model Bagus  -> Lulus: {lulus_1}, Alasan: {alasan_1}")
assert lulus_1 is True and len(alasan_1) == 0

# Skenario 2: Model Gagal Recall (seperti Gradient Boosting kita)
lulus_2, alasan_2 = check({"roc_auc": 0.767, "recall_bad": 0.3556})
print(f"Model Gagal  -> Lulus: {lulus_2}, Alasan: {alasan_2}")
assert lulus_2 is False
assert alasan_2[0] == "recall_bad 0.3556 < 0.5"

# Skenario 3: Model Gagal Semua
lulus_3, alasan_3 = check({"roc_auc": 0.60, "recall_bad": 0.40})
print(f"Model Gagal  -> Lulus: {lulus_3}, Alasan: {alasan_3}")
assert lulus_3 is False and len(alasan_3) == 2
print("✅ Fungsi `check` OK.\n")

print("=== 2. Uji Fungsi `main` (Simulasi) ===")
# Buat file dummy best_metrics.json
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
dummy_file = REPORTS_DIR / "best_metrics.json"

# Simulasi 1: Metrik lulus
with open(dummy_file, "w") as f:
    json.dump({"roc_auc": 0.75, "recall_bad": 0.60}, f)
    
exit_code = src.quality_gate.main()
assert exit_code == 0
print("✅ Fungsi `main` (Lulus) OK.\n")

# Simulasi 2: Metrik gagal
with open(dummy_file, "w") as f:
    json.dump({"roc_auc": 0.65, "recall_bad": 0.40}, f)
    
exit_code = src.quality_gate.main()
assert exit_code == 1
print("✅ Fungsi `main` (Gagal) OK.\n")

print("🎉 Tahap 3 (Quality Gate) Selesai!")

=== 1. Uji Fungsi `check` ===
Model Bagus  -> Lulus: True, Alasan: []
Model Gagal  -> Lulus: False, Alasan: ['recall_bad 0.3556 < 0.5']
Model Gagal  -> Lulus: False, Alasan: ['roc_auc 0.6 < 0.7', 'recall_bad 0.4 < 0.5']
✅ Fungsi `check` OK.

=== 2. Uji Fungsi `main` (Simulasi) ===
QUALITY GATE: LULUS
✅ Fungsi `main` (Lulus) OK.

QUALITY GATE: GAGAL -> roc_auc 0.65 < 0.7, recall_bad 0.4 < 0.5
✅ Fungsi `main` (Gagal) OK.

🎉 Tahap 3 (Quality Gate) Selesai!


### 4.Menggunakan MLflow untuk Mencatat Eksperimen

In [19]:
import importlib
import pandas as pd
import src.serve

# Reload modul
importlib.reload(src.serve)
from src.serve import CreditScorer, LOG_PATH

print("=== 1. Inisialisasi Model (Load dari MLflow) ===")
scorer = CreditScorer()
print("✅ Model champion berhasil dimuat.\n")

print("=== 2. Uji Input Valid ===")
record_valid = {
    "age": 35, "sex": "male", "job": "2", "housing": "own", 
    "saving_accounts": "little", "checking_account": "moderate", 
    "credit_amount": 5000, "duration": 24, "purpose": "car"
}
res = scorer.predict(record_valid, batch="uji_coba", request_id="req-001")
print(f"Status  : {res['status']}")
print(f"Prob Bad: {res['prob_bad']}")
print(f"Decision: {res['decision']}")
print(f"Latency : {res['latency_ms']} ms")

print("\n=== 3. Uji Input Rusak (Field Kosong) ===")
record_invalid = record_valid.copy()
record_invalid["age"] = None
res_err = scorer.predict(record_invalid, batch="uji_coba", request_id="req-002")
print(f"Status  : {res_err['status']}")
print(f"Error   : {res_err['error']}")

print("\n=== 4. Cek File Log ===")
print(f"Log disimpan di: {LOG_PATH}")
# Tampilkan 3 baris terakhir dari log
df_log = pd.read_csv(LOG_PATH)
print(df_log.tail(2).to_string())

=== 1. Inisialisasi Model (Load dari MLflow) ===
✅ Model champion berhasil dimuat.

=== 2. Uji Input Valid ===
Status  : ok
Prob Bad: 0.5914
Decision: review
Latency : 22.3 ms

=== 3. Uji Input Rusak (Field Kosong) ===
Status  : error
Error   : field kosong: ['age']

=== 4. Cek File Log ===
Log disimpan di: /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/logs/predictions.csv
                      timestamp request_id     batch status                  error  latency_ms  prob_bad decision   age  credit_amount duration   sex  job housing saving_accounts checking_account purpose
600  2026-11-09T17:56:54.898867    req-001  uji_coba     ok                    NaN       22.30    0.5914   review  35.0           5000       24  male    2     own          little         moderate     car
601  2026-11-09T17:56:54.924465    req-002  uji_coba  error  field kosong: ['age']        0.06       NaN      NaN   NaN           5000       

#### Bagian C Simulate

In [20]:
import importlib
import src.simulate

# Reload modul untuk memastikan kode terbaru yang dieksekusi
importlib.reload(src.simulate)

# Jalankan simulasi
src.simulate.main()

Menyiapkan data...
Memuat model champion...
Mengirim 300 request Minggu 1 (Normal)...
Mengirim 300 request Minggu 2 (Drift + 6 Rusak)...

✅ Label asli disimpan di /home/davidpsb/workspace/01 Advance Bootcamp AI Engineer/006 Bootcamp Babak 6/tugas_babak_6/tmp_7_DPSB/tugas_26_28_david/logs/labels.csv

=== RINGKASAN SIMULASI ===
status    error   ok
batch               
minggu_1      0  300
minggu_2      6  294


#### Bagian D Monitor

In [21]:
import importlib
import src.monitor
importlib.reload(src.monitor)

src.monitor.main()

Menyiapkan data referensi...
Menghitung drift & metrik...

=== METRIK OPERASIONAL ===
   batch  requests  error_rate  p50_latency_ms  p95_latency_ms  throughput_rps  mean_prob_bad  share_tolak    auc
minggu_1       300        0.00           16.66           19.51           52.21         0.4524       0.2733 0.7629
minggu_2       300        0.02           16.06           19.37           55.50         0.5313       0.4133 0.7387

=== DRIFT ===
   batch          feature     type      psi  ks_stat         ks_p       js  drift
minggu_1              age  numerik 0.064055 0.068095 2.727362e-01      NaN  False
minggu_1    credit_amount  numerik 0.038050 0.040952 8.591262e-01      NaN  False
minggu_1         duration  numerik 0.012336 0.041429 8.497101e-01      NaN  False
minggu_1              sex kategori      NaN      NaN          NaN 0.034757  False
minggu_1              job kategori      NaN      NaN          NaN 0.053810  False
minggu_1          housing kategori      NaN      NaN          NaN

#### Menguji Semua tets

In [22]:
import subprocess
import sys

print("🔄 Menjalankan SEMUA test (8 test)...")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)
print(result.stdout)

if result.returncode == 0:
    print("🎉🎉🎉 SEMUA 8 TEST LULUS! Tugas 100% SELESAI!")
else:
    print("⚠️ Masih ada test yang gagal. Cek output di atas.")

🔄 Menjalankan SEMUA test (8 test)...
........                                                                 [100%]
8 passed in 5.99s

🎉🎉🎉 SEMUA 8 TEST LULUS! Tugas 100% SELESAI!
